# ai-checker - Training Pipeline

In [ ]:
# 1. 从 GitHub 克隆仓库
!git clone https://github.com/dylanzihao/ai-checker.git /kaggle/working/ai-checker

In [ ]:
# 2. 安装 Kaggle 训练依赖
!pip install -q -r /kaggle/working/ai-checker/requirements/requirements-kaggle.txt
!pip freeze | grep -E 'torch|transformers|datasets|accelerate'

In [ ]:
# 2.5 预加载模型并转为 safetensors 缓存，避免训练子进程 I/O 卡顿
import os
from transformers import AutoModel, AutoConfig, AutoTokenizer

MODEL_INPUT_DIR = "/kaggle/input/models/dylanzihao/chinese-bert-wwm-ext/transformers/default/1"
MODEL_CACHE_DIR = "/kaggle/working/models/pretrained"

if not os.path.exists(MODEL_CACHE_DIR):
    print(f"从 {MODEL_INPUT_DIR} 加载预训练模型...")
    model = AutoModel.from_pretrained(MODEL_INPUT_DIR, local_files_only=True)
    config = AutoConfig.from_pretrained(MODEL_INPUT_DIR, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_INPUT_DIR, local_files_only=True)

    print(f"保存为 safetensors 至 {MODEL_CACHE_DIR}...")
    os.makedirs(MODEL_CACHE_DIR, exist_ok=True)
    model.save_pretrained(MODEL_CACHE_DIR, safe_serialization=True)
    config.save_pretrained(MODEL_CACHE_DIR)
    tokenizer.save_pretrained(MODEL_CACHE_DIR)
    print("模型缓存完成。")
else:
    print(f"模型缓存已存在: {MODEL_CACHE_DIR}")

In [ ]:
# 3. 启动分布式训练
import subprocess
import sys
import torch
import os

n_gpus = torch.cuda.device_count()
print(f"可用 GPU: {n_gpus}")

if n_gpus == 0:
    raise SystemExit("没有 GPU，请在 Kaggle 设置中开启 GPU 加速器后重跑。")

if n_gpus > 1:
    cmd = [
        sys.executable, '-m', 'torch.distributed.run',
        f'--nproc_per_node={n_gpus}',
        '--master_port=29500',
        '/kaggle/working/ai-checker/src/train/trainer.py',
    ]
else:
    cmd = [sys.executable, '/kaggle/working/ai-checker/src/train/trainer.py']

result = subprocess.run(cmd)
print(f"\nreturncode: {result.returncode}")